In [ ]:
import glob
import os
import re
import sys

import numpy as np
import rasterio
import torch
from matplotlib import pyplot as plt
from matplotlib.colors import Normalize
from matplotlib import cm

_gap_aware_root = os.environ.get(
    "GAP_AWARE_ROOT",
    os.path.abspath(os.path.join(os.path.dirname("__file__"), "..", "..", ".."))
)
if _gap_aware_root not in sys.path:
    sys.path.insert(0, _gap_aware_root)

from model_gap_aware.maskfeat import MaskFeatGapFiller
from sentinel.src.evaluation.evaluate_gapfilling import GapFillingEvalDataset, find_checkpoint

# ── config ──────────────────────────────────────────────────────────────────
_s2_data_dir = os.environ.get(
    "S2_DATA_DIR",
    os.path.join(_gap_aware_root, "sentinel", "data"),
)
gap_fill_root = os.path.join(_s2_data_dir, "gap_filling")
log_dir       = os.path.join(_gap_aware_root, "sentinel", "lightning_logs/experiments/exp01_zero_shot/")
checkpoint    = None   # set to a .ckpt path to override auto-discovery
device        = "cpu"
normalize     = True  # must match the normalization used during training
n_samples     = 5
seed          = None

n_timesteps = 5
MIDDLE_T    = n_timesteps // 2   # index 2
RGB_CHANNELS = [2, 1, 0]         # B4 (Red), B3 (Green), B2 (Blue) — rasterio bands 3, 2, 1

# ── load model ──────────────────────────────────────────────────────────────
ckpt_path = find_checkpoint(log_dir, checkpoint)
print(f"Loading checkpoint: {ckpt_path}")
model = MaskFeatGapFiller.load_from_checkpoint(ckpt_path, map_location=device)
model.to(device)
model.eval()
PATCH_SIZE = model.patch_size

# ── helpers ──────────────────────────────────────────────────────────────────
def robust_rgb(
    b4: np.ndarray, b3: np.ndarray, b2: np.ndarray,
    pmin: float = 2, pmax: float = 98,
    lo: float = None, hi: float = None,
):
    rgb  = np.stack([b4, b3, b2], axis=-1).astype(np.float32)
    gaps = np.any(np.isnan(rgb), axis=-1)
    if lo is None or hi is None:
        lo = np.nanpercentile(rgb, pmin)
        hi = np.nanpercentile(rgb, pmax)
    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        lo, hi = 0.0, 2500.0
    rgb = (rgb - lo) / (hi - lo + 1e-6)
    rgb = np.clip(rgb, 0, 1)
    rgb[gaps] = 0.0
    return rgb, gaps


def read_raw_rgb(timestep_dir: str):
    """Return (b4, b3, b2) raw float32 arrays from S2C.tif without normalization."""
    try:
        with rasterio.open(os.path.join(timestep_dir, "S2C.tif")) as src:
            b2 = src.read(1).astype(np.float32)
            b3 = src.read(2).astype(np.float32)
            b4 = src.read(3).astype(np.float32)
        return b4, b3, b2
    except Exception:
        dummy = np.full((64, 64), np.nan)
        return dummy, dummy.copy(), dummy.copy()


def read_rgb(timestep_dir: str, lo: float = None, hi: float = None):
    """Return (rgb (H,W,3), gaps (H,W bool)) from multi-band S2C.tif."""
    try:
        b4, b3, b2 = read_raw_rgb(timestep_dir)
        rgb, gaps = robust_rgb(b4, b3, b2, lo=lo, hi=hi)
        return rgb, gaps
    except Exception:
        dummy = np.zeros((64, 64, 3), dtype=np.float32)
        return dummy, np.zeros((64, 64), dtype=bool)


def date_label(folder_name: str) -> str:
    d = folder_name[:8]
    try:
        return f"{d[:4]}/{d[4:6]}/{d[6:8]}"
    except Exception:
        return d


def tensor_to_rgb(t: torch.Tensor, lo: float = None, hi: float = None, dn_scale: float = 1.0) -> tuple:
    """Convert (C, H, W) tensor to display RGB. dn_scale converts tensor values to DN space."""
    arr = t.cpu().numpy()
    r, g, b = RGB_CHANNELS
    rgb, gaps = robust_rgb(arr[r] * dn_scale, arr[g] * dn_scale, arr[b] * dn_scale, lo=lo, hi=hi)
    return rgb, gaps


def compute_gap_patch_mask(frame, patch_size):
    """frame: (C, H, W) or (H, W) — NaN at gaps. Returns (H, W) bool, True = patch masked."""
    if frame.ndim == 2:
        frame = frame[np.newaxis]
    C, H, W = frame.shape
    n_ph, n_pw = H // patch_size, W // patch_size
    r = frame.reshape(C, n_ph, patch_size, n_pw, patch_size)
    patch_has_nan = np.isnan(r).any(axis=(0, 2, 4))
    return np.repeat(np.repeat(patch_has_nan, patch_size, axis=0), patch_size, axis=1).astype(bool)


# ── dataset + reservoir sampling ────────────────────────────────────────────
dataset = GapFillingEvalDataset(gap_fill_root, normalize=normalize)

rng = np.random.default_rng(seed)
chosen_indices = sorted(rng.choice(len(dataset), size=min(n_samples, len(dataset)), replace=False).tolist())

# ── run inference and collect display data ───────────────────────────────────
ssl_dir = os.path.join(gap_fill_root, "SSL-consecutive")
gt_dir  = os.path.join(gap_fill_root, "ground_truth")

samples = []
total_gap_px = 0
total_err    = 0.0

with torch.no_grad():
    for idx in chosen_indices:
        input_np, gt_np, gap_mask_np = dataset[idx]  # numpy arrays

        x        = torch.from_numpy(input_np).unsqueeze(0).to(device)    # (1, T, C, H, W)
        gt       = torch.from_numpy(gt_np).unsqueeze(0).to(device)       # (1, C, H, W)
        gap_mask = torch.from_numpy(gap_mask_np).unsqueeze(0).to(device)  # (1, H, W)

        pred, _ = model(x, gap_only_masking=True)  # (1, C, H, W)

        # Exclude NaN GT pixels (GEE edge/boundary pixels) from error computation
        gt_valid  = ~torch.isnan(gt)                                          # (1, C, H, W)
        gt_clean  = torch.nan_to_num(gt, nan=0.0)
        loss_mask = (gap_mask.unsqueeze(1).bool() & gt_valid).float()         # (1, C, H, W)

        err       = torch.abs(pred - gt_clean)                                # (1, C, H, W)
        # per-pixel mean error over valid channels, for visualization
        valid_ch  = loss_mask.sum(dim=1).squeeze(0).clamp(min=1)              # (H, W)
        err_mean  = (err * loss_mask).sum(dim=1).squeeze(0) / valid_ch        # (H, W)
        mask_hw   = gap_mask.squeeze(0)                                        # (H, W)

        n_gap        = loss_mask.sum().item()
        total_gap_px += n_gap
        total_err    += (err * loss_mask).sum().item()

        # Paths for reading the full SSL-consecutive time series for display
        loc_id     = dataset.locations[idx]
        ssl_sample = os.path.join(ssl_dir, loc_id)
        gt_sample  = os.path.join(gt_dir, loc_id)

        ts_dirs    = sorted(d for d in glob.glob(os.path.join(ssl_sample, "*")) if os.path.isdir(d))
        gt_ts_dirs = sorted(d for d in glob.glob(os.path.join(gt_sample, "*")) if os.path.isdir(d))

        samples.append({
            "loc_id":      loc_id,
            "ts_dirs":     ts_dirs,
            "gt_ts_dirs":  gt_ts_dirs,
            "pred":        pred.squeeze(0),    # (C, H, W)
            "err_mean":    err_mean,            # (H, W)
            "gap_mask":    mask_hw,             # (H, W)
            "input_np":    input_np,            # (T, C, H, W) for gap mask computation
        })

C = pred.shape[1]
overall_mae = total_err / max(total_gap_px, 1)
print(f"Mean MAE over gap pixels ({len(samples)} samples): {overall_mae:.4f}")

# ── plot ─────────────────────────────────────────────────────────────────────
# columns: t0 | t1 | t2 (gappy) | t3 | t4 | GT clean | Prediction | Error map
N_COLS   = n_timesteps + 3
n_found  = len(samples)
fig, axes = plt.subplots(2 * n_found, N_COLS, figsize=(3.5 * N_COLS, 4.2 * 2 * n_found), squeeze=False)

# scale factor to convert model tensor values back to raw DN space for display
dn_scale = 10000.0 if normalize else 1.0

for row, s in enumerate(samples):
    row_in   = 2 * row
    row_mask = 2 * row + 1

    ts_dirs    = s["ts_dirs"]
    gt_ts_dirs = s["gt_ts_dirs"]

    # -- compute shared lo/hi from all TIF panels in this row --
    all_raw = []
    for tdir in ts_dirs + gt_ts_dirs:
        for band_arr in read_raw_rgb(tdir):
            valid = band_arr[~np.isnan(band_arr)]
            if valid.size:
                all_raw.append(valid)
    if all_raw:
        combined = np.concatenate(all_raw)
        row_lo = float(np.percentile(combined, 2))
        row_hi = float(np.percentile(combined, 98))
    else:
        row_lo, row_hi = 0.0, 2500.0
    if not np.isfinite(row_lo) or not np.isfinite(row_hi) or row_hi <= row_lo:
        row_lo, row_hi = 0.0, 2500.0

    # -- time series (t0 .. t4) --
    for t in range(n_timesteps):
        ax = axes[row_in][t]
        if t >= len(ts_dirs):
            ax.set_visible(False)
            continue
        rgb, gaps = read_rgb(ts_dirs[t], lo=row_lo, hi=row_hi)
        frac = 100.0 * float(gaps.sum()) / max(gaps.size, 1)
        label = "gappy" if t == MIDDLE_T else ""
        title = f"t={t}{' (' + label + ')' if label else ''}\n{date_label(os.path.basename(ts_dirs[t]))}\ngaps={frac:.1f}%"
        ax.imshow(rgb)
        ax.set_title(title, fontsize=7)
        ax.axis("off")

    # -- ground truth --
    ax_gt = axes[row_in][n_timesteps]
    if gt_ts_dirs:
        gt_rgb, _ = read_rgb(gt_ts_dirs[0], lo=row_lo, hi=row_hi)
        ax_gt.imshow(gt_rgb)
        ax_gt.set_title(f"GT (t=2)\n{date_label(os.path.basename(gt_ts_dirs[0]))}\nclean", fontsize=7)
        for spine in ax_gt.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor("#1a7f1a")
            spine.set_linewidth(2)
    else:
        ax_gt.text(0.5, 0.5, "GT missing", ha="center", va="center", transform=ax_gt.transAxes)
    ax_gt.axis("off")

    # -- prediction --
    ax_pred = axes[row_in][n_timesteps + 1]
    pred_rgb, _ = tensor_to_rgb(s["pred"], lo=row_lo, hi=row_hi, dn_scale=dn_scale)
    if MIDDLE_T < len(s["ts_dirs"]):
        input_mid_rgb, input_mid_gaps = read_rgb(s["ts_dirs"][MIDDLE_T], lo=row_lo, hi=row_hi)
    else:
        input_mid_rgb = np.zeros_like(pred_rgb)
        input_mid_gaps = np.zeros((pred_rgb.shape[0], pred_rgb.shape[1]), dtype=bool)
    gap_np = s["gap_mask"].cpu().numpy().astype(bool)
    fill_np = gap_np | input_mid_gaps
    merged_rgb = np.where(fill_np[:, :, np.newaxis], pred_rgb, input_mid_rgb)
    ax_pred.imshow(merged_rgb)
    ax_pred.set_title("Prediction (t=2)\ngap+NaN px merged", fontsize=7)
    for spine in ax_pred.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor("#1a1a8f")
        spine.set_linewidth(2)
    ax_pred.axis("off")

    # -- error map (gap pixels only) --
    ax_err = axes[row_in][n_timesteps + 2]
    err_np   = s["err_mean"].cpu().numpy()    # (H, W)
    mask_np  = s["gap_mask"].cpu().numpy()    # (H, W)
    err_disp = np.where(mask_np > 0, err_np, np.nan)
    vmax     = np.nanpercentile(err_disp, 95) if np.any(~np.isnan(err_disp)) else 1.0
    im       = ax_err.imshow(err_disp, cmap="viridis", vmin=0, vmax=vmax)
    coverage = 100.0 * float(mask_np.sum()) / max(mask_np.size, 1)
    ax_err.set_title(f"Error (gap px)\ncoverage={coverage:.1f}%", fontsize=7)
    ax_err.axis("off")
    plt.colorbar(im, ax=ax_err, fraction=0.046, pad=0.04)

    axes[row_in][0].set_ylabel(f"sample\n{s['loc_id']}", fontsize=8, labelpad=4)
    axes[row_in][0].yaxis.set_visible(True)

    # -- masked-view row (gap masking only — gap_only_masking=True) --
    r, g, b = RGB_CHANNELS
    for t in range(n_timesteps):
        ax = axes[row_mask][t]
        if t < len(ts_dirs):
            frame = s["input_np"][t]  # (C, H, W) normalized
            pixel_mask = compute_gap_patch_mask(frame, PATCH_SIZE)
            rgb_m, _ = read_rgb(ts_dirs[t], lo=row_lo, hi=row_hi)
            rgb_m[pixel_mask] = 0.5
            ax.imshow(rgb_m)
            pct = 100.0 * pixel_mask.mean()
            label = "gappy" if t == MIDDLE_T else ""
            title = f"t={t}{' (' + label + ')' if label else ''}\nmasked={pct:.1f}%"
            ax.set_title(title, fontsize=7)
        ax.axis("off")
    for col in range(n_timesteps, N_COLS):
        axes[row_mask][col].axis("off")
    axes[row_mask][0].set_ylabel("masked view", fontsize=7, labelpad=4)
    axes[row_mask][0].yaxis.set_visible(True)

fig.suptitle(
    f"Zero-shot gap-filling evaluation  ({n_found} samples)  |  "
    f"Mean MAE (gap px) = {overall_mae:.4f}\n"
    "Green border = GT clean  |  Blue border = pred merged with input (NaN→model)  |  "
    "Error map shows mean |pred−GT| over synthetic gap pixels (viridis)  |  "
    "Masked-view row: gray = patch masked by encoder (gap masking)",
    fontsize=10,
    y=1.01,
)
plt.tight_layout()
plt.show()